## **OPTIMIZACIÓN DE PARÁMETROS EGNA**

#### **NLSE**

In [1]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.NonLinearSE import *
import time
#Para Bayesian optimization de parámetros
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA
#Para procesos Gaussianos
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import pickle

n_variables = [50]     #[50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)
##################################################################
######### MANDATORY ARCS for GBN structure #######################
##################################################################
if n_variables[0]%2 ==0:
    varsNLSE=n_variables[0]+1
else:
    varsNLSE=n_variables[0]
#ARCOS cadena simple
#whites es una lista con los arcos obligatorios que aprender
#'Continuidad de la función'
chain=[(str(i), str(i+1)) for i in range(varsNLSE-1)]

#ARCOS [(0, 50), (1, 49), (2, 48), ..., (23, 27), (24, 26)] conectando puntos extremos (simetría)
# #simetría de la función SOLO PARA OSCILADOR SI ACASO
extremes=[(str(i), str(varsNLSE-i-1)) for i in range((varsNLSE//2))]

#ARCOS de evidencias a resto centro fuera para NLSE
#simetría de función desde punto central (condición de contorno)
whites1=[(str(i), str(i-1)) for i in range((varsNLSE//2),0, -1)]
whites2=[(str(i), str(i+1)) for i in range(varsNLSE//2,varsNLSE-1, 1)]
centertoout = whites1 + whites2

#ARCOS de evidencias a resto fuera dentro para Well y Osc
#simetría de función desde punto central (condición de contorno)
whites1=[(str(i), str(i+1)) for i in range(0,varsNLSE//2, 1)]
whites2=[(str(i), str(i-1)) for i in range(varsNLSE-1, (varsNLSE//2),-1)]
outtocenter = whites1 + whites2

ARCOS = {}
ARCOS['chain'] = chain
ARCOS['extremes'] = extremes
ARCOS['centertoout'] = centertoout
ARCOS['outtocenter'] = outtocenter
ARCOS['None'] = None #[chain, extremes, centertoout, None]
##################################################################
##################################################################

search_space = [
    Categorical([50, 500, 1000, 1500, 2000, 2500, 3000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Real(0.1, 0.8, name="alpha", prior='uniform'),                # Continuous values
    Real(0.1, 0.8, name="elite_factor", prior='uniform'),        # Continuous values
    Categorical(['chain', 'extremes', 'centertoout', 'outtocenter' ,'None'], name="whites")      # Discrete values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor, whites):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the mean MSE (MSE across the four first states of optimization problem).
    """
    ######################################################################################
    ######################################################################################
    #################### N L S E Non linear Schroedinger Equation ########################
    ######################################################################################
    print('\nNLSE IN PROGRESS\n')

    Resultados = np.empty((1, 9, 5)) 
    # 5 métricas [MSE, E, E%, GP_1000, t_exc]
    experimento = int(0)   
    L=2
    #El grid se define en para los tres problemas de optimización con el mismo número de variables
    if v%2 ==0:
        varsNLSE=v+1
    else:
        varsNLSE=v

    X_grid = np.linspace(start=-1, stop=1, num=varsNLSE).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 que son 1
    #X_train = X_grid[n_variables//2]
    X_train = np.array((0)).reshape(-1,1)
    Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
    
    init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
    params = {'L':L}
    #qegna = UMDAc(size_gen=3000, max_iter=600, dead_iter=201, n_variables=varsNLSE, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= varsNLSE, lower_bound=0, upper_bound = 2, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egparesultNLSE = qegna.minimize(NLSE, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 8, 4] = end-start
    MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.sel_inds_hist[-1, 0, :-1])**2) / len(X_grid[:,0])
    MSE_NLSE = np.float64(MSE_NLSE)
    Resultados[experimento, 8, 0] = MSE_NLSE
    #setting non used columns to None
    Resultados[experimento, 8, 1:3] = None
    Resultados[experimento, :, 3] = None
    print(f'MSE: {MSE_NLSE}\n')    
    print('\nNLSE DONE\n')
    return MSE_NLSE


global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

for v in n_variables:
    # Run Bayesian Optimization
    print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
    result = gp_minimize(
        func=objective,      # Objective function
        dimensions=search_space,  # Search space
        n_calls=50,         # Number of function evaluations
        random_state=42,     # For reproducibility,
        n_initial_points=12
    )

    # Best parameters found
    best_params = dict(zip(['size_gen', 'alpha', 'elite_factor', 'whites'], result.x))
    print("Best Parameters:", best_params)
    print("Best MSE (Min-Max Approach):", result.fun)


    
    ruta = f'EGNABO_params/{v}NLSE'
    with open(f'{ruta}.pkl', "wb") as file:
        pickle.dump(best_params, file)


    for keys in best_params.keys():
        print(f'Problema {keys}: {best_params[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 50
##############################################



NLSE IN PROGRESS

IT:  0 	Best cost:  1268379.5907898635337
	NFEVALS = 290000 F = -0.3610731892671690213
	X = [[0.69981599 0.71909019 0.73824507 0.75719295 0.77601398 0.79467944
  0.81304422 0.83066555 0.84777574 0.86435986 0.88026078 0.89539193
  0.90962658 0.92284388 0.93504274 0.94615489 0.95611493 0.96481807
  0.97229578 0.97838796 0.98294867 0.98614661 0.987842   0.98802677
  0.9866824  0.98398048 0.97988612 0.97441355 0.96746865 0.95924816
  0.94983585 0.93917332 0.92738908 0.91456474 0.90075884 0.88609351
  0.87055979 0.85430348 0.83728472 0.81967569 0.80172826 0.7834857
  0.76511129 0.74661314 0.72813327 0.70954919 0.69100688 0.67248632
  0.65413254 0.63601068 0.61813189]]
MSE: 0.001185226653394599


NLSE DONE


################################

## **Infinite Well**

In [2]:
def _normalize(dataset:np.array, grid_size: float):
        '''
        Normalizes the solutions i.e. multiply by a constant so that the integral of the solution^2 = 1.
        Be aware that integration depends on problem definition and size!!!!
        '''
        #grid_size = L/variables     dx of integral
        #grid_size = 5/(self.n_variables-1)

        #Assuming a constant size grid we know grid size independently of problem parameters (better code-wise)
        #self.grid_size = grid_X[-1,0] - grid_X[-2, 0]

        for ind in range(len(dataset)):
                integral = 0
                for var in range(dataset.shape[1]):
                    integral = integral + (dataset[ind,:][var]**2)*grid_size ##Tener en cuenta posibles cambios en tamaño grid!!! L caja = 5 

                A = (1/integral)**0.5
                dataset[ind, :] = A * dataset[ind, :]


In [3]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDwellfunctions import *
import time
#Para Bayesian optimization de parámetros
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA
#Para procesos Gaussianos
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import pickle

n_variables = [50]     #[50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)
##################################################################
######### MANDATORY ARCS for GBN structure #######################
##################################################################
#ARCOS cadena simple
#whites es una lista con los arcos obligatorios que aprender
#'Continuidad de la función'
chain=[(str(i), str(i+1)) for i in range(n_variables[0]-1)]

#ARCOS [(0, 50), (1, 49), (2, 48), ..., (23, 27), (24, 26)] conectando puntos extremos (simetría)
# #simetría de la función SOLO PARA OSCILADOR SI ACASO
extremes=[(str(i), str(n_variables[0]-i-1)) for i in range((n_variables[0]//2))]

#ARCOS de evidencias a resto centro fuera para NLSE
#simetría de función desde punto central (condición de contorno)
whites1=[(str(i), str(i-1)) for i in range((n_variables[0]//2),0, -1)]
whites2=[(str(i), str(i+1)) for i in range(n_variables[0]//2,n_variables[0]-1, 1)]
centertoout = whites1 + whites2

#ARCOS de evidencias a resto fuera dentro para Well y Osc
#simetría de función desde punto central (condición de contorno)
whites1=[(str(i), str(i+1)) for i in range(0,n_variables[0]//2, 1)]
whites2=[(str(i), str(i-1)) for i in range(n_variables[0]-1, (n_variables[0]//2),-1)]
outtocenter = whites1 + whites2

ARCOS = {}
ARCOS['chain'] = chain
ARCOS['extremes'] = extremes
ARCOS['centertoout'] = centertoout
ARCOS['outtocenter'] = outtocenter
ARCOS['None'] = None #[chain, extremes, centertoout, None]
##################################################################
##################################################################

search_space = [
    Categorical([50, 500, 1000, 1500, 2000, 2500, 3000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Real(0.1, 0.8, name="alpha", prior='uniform'),                # Continuous values
    Real(0.1, 0.8, name="elite_factor", prior='uniform'),        # Continuous values
    Categorical(['chain', 'extremes', 'centertoout', 'outtocenter' ,'None'], name="whites")      # Discrete values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor, whites):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the mean MSE (MSE across the four first states of optimization problem).
    """
    Resultados = np.empty((1, 9, 5)) 
    # 5 métricas [MSE, E, E%, GP_1000, t_exc]
    experimento = int(0)    

    ######################################################################################
    ######################################################################################
    ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
    ######################################################################################

    print('\nInfinite Well IN PROGRESS\n')
    L=5
    #Energías teóricas
    Eg = (2*np.pi)**2/(8*L**2*1)
    En=[]
    for i in range(4):
        e=(i+1)**2*Eg
        En.append(e)

    ## Problem grid definition
    X_grid = np.linspace(start=0, stop=L, num=v).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
    init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
    grid_size = X_grid[-1,0] - X_grid[-2, 0]
    params = {'L':L}

    #Soluciones teóricas
    ### Soluciones analíticas
    # def Psi_n(n):
    #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

    APsisWell = {}
    for n in range(4):
        APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, v)/L)
    ###############################    
    #Ground state n=1
    ###############################
    print(f'F should be approx -0.9618')
    #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultgroundWell = qegna.minimize(oneD_wellExpE, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 4, 4] = end-start
    _normalize(dataset=egnaresultgroundWell.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_fund=egnaresultgroundWell.sel_inds_hist[-1, 0, :-1]
    E_fund = np.sqrt(-np.log(-egnaresultgroundWell.sel_inds_hist[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
    #En este caso no hay constante multiplicando exponente exponencial
    Resultados[experimento, 4, 1] = E_fund
    Resultados[experimento, 4, 2] = abs(E_fund-En[0])/En[0]*100 

    params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
    print(f'\n  |Psi0> done\n')

    ###############################   
    #1excitated state n=2
    ###############################   
    print(f'F should be approx -0.7042')
    #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultWellexc1 = qegna.minimize(oneD_wellExpEexcitated, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 5, 4] = end-start
    _normalize(dataset=egnaresultWellexc1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_1=egnaresultWellexc1.sel_inds_hist[-1, 0, :-1]
    E1 = np.sqrt(-np.log(-egnaresultWellexc1.sel_inds_hist[-1,0,-1]))
    Resultados[experimento, 5, 1] = E_fund + E1
    Resultados[experimento, 5, 2] = abs(E_fund + E1 -En[1])/En[1]*100

    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
    print(f'\n  |Psi1> done\n')

    ###############################   
    #2nd excitated state n=3
    ###############################   
    print(f'F should be approx -0.3775')
    #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultWellexc2 = qegna.minimize(oneD_wellExpEexcitated3, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 6, 4] = end - start
    _normalize(dataset=egnaresultWellexc2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_2=egnaresultWellexc2.sel_inds_hist[-1, 0, :-1]
    E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))
    Resultados[experimento, 6, 1] = E_fund + E1 + E2
    Resultados[experimento, 6, 2] = abs(E_fund + E1 +E2 -En[2])/En[2]*100

    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
    print(f'\n  |Psi2> done\n')

    ###############################   
    #3rd excitated state n=4
    ###############################   
    print(f'F should be approx -0.1482')
    #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultWellexc3 = qegna.minimize(oneD_wellExpEexcitated4, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 7, 4] = end-start
    _normalize(dataset=egnaresultWellexc3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_3 = egnaresultWellexc3.sel_inds_hist[-1, 0, :-1]
    E3 = np.sqrt(-np.log(-egnaresultWellexc3.sel_inds_hist[-1,0,-1]))
    Resultados[experimento, 7, 1] = E_fund + E1 + E2 + E3
    Resultados[experimento, 7, 2] = abs(E_fund + E1 +E2+E3 -En[3])/En[3]*100

    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = min(np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2))
        Resultados[experimento, i+4, 0] = mse
        MSEs.append(mse)

    MSEmean = np.mean(MSEs)
    print('\nInfinite Well DONE\n')
    return MSEmean


global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

for v in n_variables:
    # Run Bayesian Optimization
    print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
    result = gp_minimize(
        func=objective,      # Objective function
        dimensions=search_space,  # Search space
        n_calls=50,         # Number of function evaluations
        random_state=42,     # For reproducibility,
        n_initial_points=12
    )

    # Best parameters found
    best_params = dict(zip(['size_gen', 'alpha', 'elite_factor', 'whites'], result.x))
    print("Best Parameters:", best_params)
    print("Best MSE (Min-Max Approach):", result.fun)


    
    ruta = f'EGNABO_params/{v}Well'
    with open(f'{ruta}.pkl', "wb") as file:
        pickle.dump(best_params, file)


    for keys in best_params.keys():
        print(f'Problema {keys}: {best_params[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 50
##############################################



Infinite Well IN PROGRESS

F should be approx -0.9618
IT:  0 	Best cost:  2491.411546277125
	NFEVALS = 375000 F = -0.6292365354409827
	X = [[-3.80418309e-01 -1.38456203e-03 -3.25810031e-01 -3.50030575e-01
  -3.57360518e-01 -2.85124137e-01 -1.89093478e-01 -2.81670528e-01
  -3.42418167e-01 -4.58277862e-01 -5.39379142e-01 -5.46804634e-01
  -6.20898381e-01 -6.03742936e-01 -6.36783329e-01 -5.36574870e-01
  -5.11165804e-01 -5.58033280e-01 -5.25238822e-01 -5.71973777e-01
  -5.74241084e-01 -5.69243514e-01 -5.70385681e-01 -4.98475667e-01
  -6.56562355e-01 -6.22748082e-01 -6.68726585e-01 -6.57647250e-01
  -6.17863438e-01 -5.68997578e-01 -5.73097432e-01 -4.71348073e-01
  -4.55081026e-01 -4.15101123e-01 -4.29279750e-01 -3.71099444e-01
  -3.13700512e-01 -3.39668470

## **Quantum Oscillator**

In [4]:
def _normalize(dataset:np.array, grid_size: float):
        '''
        Normalizes the solutions i.e. multiply by a constant so that the integral of the solution^2 = 1.
        Be aware that integration depends on problem definition and size!!!!
        '''
        #grid_size = L/variables     dx of integral
        #grid_size = 5/(self.n_variables-1)

        #Assuming a constant size grid we know grid size independently of problem parameters (better code-wise)
        #self.grid_size = grid_X[-1,0] - grid_X[-2, 0]

        for ind in range(len(dataset)):
                integral = 0
                for var in range(dataset.shape[1]):
                    integral = integral + (dataset[ind,:][var]**2)*grid_size ##Tener en cuenta posibles cambios en tamaño grid!!! L caja = 5 

                A = (1/integral)**0.5
                dataset[ind, :] = A * dataset[ind, :]


In [5]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDQHarmOscillatorfunctions import *
import time
#Para Bayesian optimization de parámetros
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA
#Para procesos Gaussianos
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import pickle

n_variables = [50]     #[50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)
##################################################################
######### MANDATORY ARCS for GBN structure #######################
##################################################################
#ARCOS cadena simple
#whites es una lista con los arcos obligatorios que aprender
#'Continuidad de la función'
chain=[(str(i), str(i+1)) for i in range(n_variables[0]-1)]

#ARCOS [(0, 50), (1, 49), (2, 48), ..., (23, 27), (24, 26)] conectando puntos extremos (simetría)
# #simetría de la función SOLO PARA OSCILADOR SI ACASO
extremes=[(str(i), str(n_variables[0]-i-1)) for i in range((n_variables[0]//2))]

#ARCOS de evidencias a resto centro fuera para NLSE
#simetría de función desde punto central (condición de contorno)
whites1=[(str(i), str(i-1)) for i in range((n_variables[0]//2),0, -1)]
whites2=[(str(i), str(i+1)) for i in range(n_variables[0]//2,n_variables[0]-1, 1)]
centertoout = whites1 + whites2

#ARCOS de evidencias a resto fuera dentro para Well y Osc
#simetría de función desde punto central (condición de contorno)
whites1=[(str(i), str(i+1)) for i in range(0,n_variables[0]//2, 1)]
whites2=[(str(i), str(i-1)) for i in range(n_variables[0]-1, (n_variables[0]//2),-1)]
outtocenter = whites1 + whites2

ARCOS = {}
ARCOS['chain'] = chain
ARCOS['extremes'] = extremes
ARCOS['centertoout'] = centertoout
ARCOS['outtocenter'] = outtocenter
ARCOS['None'] = None #[chain, extremes, centertoout, None]
##################################################################
##################################################################

search_space = [
    Categorical([50, 500, 1000, 1500, 2000, 2500, 3000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Real(0.1, 0.8, name="alpha", prior='uniform'),                # Continuous values
    Real(0.1, 0.8, name="elite_factor", prior='uniform'),        # Continuous values
    Categorical(['chain', 'extremes', 'centertoout', 'outtocenter' ,'None'], name="whites")      # Discrete values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor, whites):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the mean MSE (MSE across the four first states of optimization problem).
    """
    Resultados = np.empty((1, 9, 5)) 
    # 5 métricas [MSE, E, E%, GP_1000, t_exc]
    experimento = int(0) 
    ##########################################################################################
    ##########################################################################################
    ########################### OSCILADOR ARMÓNICO ###########################################
    ##########################################################################################
    print('Quantum Harmonic Oscillator IN PROGRESS\n')
    #Problem grid definition
    X_grid = np.linspace(start=-4, stop=4, num=v).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

    #Analytical solutions definitions
    #Ground state
    APsi0 = np.exp(-X_grid**2/2)
    integral =0
    for i in range(len(APsi0)):
        integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

    B = (1/integral)**0.5
    APsi0 = B * APsi0

    #1st excitated state
    APsi1 = np.exp(-X_grid**2/2)*2*X_grid
    integral =0
    for i in range(len(APsi1)):
        integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

    B = (1/integral)**0.5
    APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

    #2nd excitated state
    APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
    integral =0
    for i in range(len(APsi2)):
        integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

    B = (1/integral)**0.5
    APsi2 = B * APsi2

    #3rd excitated state
    APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
    integral =0
    for i in range(len(APsi3)):
        integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

    B = (1/integral)**0.5
    APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

    APsis = {}
    APsis['APsi0'] = APsi0[:, 0]
    APsis['APsi1'] = APsi1[:, 0]
    APsis['APsi2'] = APsi2[:, 0]
    APsis['APsi3'] = APsi3[:, 0]
    X_grid = X_grid[:, 0]
    grid_size = X_grid[-1] - X_grid[-2]

    ##########################################################################################################
    ##########################################################################################################

    ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
    params = {'x':X_grid}
    #Ground state
    print(f'F should be approx -0.082')
    #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHOground = qegna.minimize(oneDHOExpE, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 0, 4] = end-start
    _normalize(dataset=egnaresultQHOground.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    #Se guardan resultados de algoritmo para siguiente estado excitado
    Psi_ground=egnaresultQHOground.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E0 = np.sqrt(-np.log(-egnaresultQHOground.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
    Resultados[experimento, 0, 1] = E0
    Resultados[experimento, 0, 2] = abs(E0-0.5)/0.5*100

    print(f'\n  |Psi0> done\n')

    #1er estado excitado
    print(f'F should be approx -4.54e-5')
    #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHO1 = qegna.minimize(oneDHOExpE1, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 1, 4] = end-start
    _normalize(dataset=egnaresultQHO1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi1=egnaresultQHO1.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E1 = np.sqrt(-np.log(-egnaresultQHO1.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    Resultados[experimento, 1, 1] = E0 + E1
    Resultados[experimento, 1, 2] = abs(E0 + E1 -1.5)/1.5*100
    
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
    print(f'\n  |Psi1> done\n')

    #2o estado excitado
    print(f'F should be approx -4.54e-5')
    #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHO2 = qegna.minimize(oneDHOExpE2, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 2, 4] = end-start
    _normalize(dataset=egnaresultQHO2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi2=egnaresultQHO2.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    Resultados[experimento, 2, 1] = E0 + E1 + E2
    Resultados[experimento, 2, 2] = abs(E0 + E1 + E2 -2.5)/2.5*100
    
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
    print(f'\n  |Psi2> done\n')

    #3er estado excitado
    print(f'F should be approx -4.54e-5')
    #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=v, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
    qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHO3 = qegna.minimize(oneDHOExpE3, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 3, 4] = end-start
    _normalize(dataset=egnaresultQHO3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi3=egnaresultQHO3.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E3 = np.sqrt(-np.log(-egnaresultQHO3.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    Resultados[experimento, 3, 1] = E0 + E1 + E2 + E3
    Resultados[experimento, 3, 2] = abs(E0 + E1 + E2 + E3 -3.5)/3.5*100
    
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = min(np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsis[f'APsi{i}'])**2))
        Resultados[experimento, i, 0] = mse
        MSEs.append(mse)

    MSEmean = np.mean(MSEs)
    print('\nInfinite Oscillator DONE\n')
    return MSEmean


global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

for v in n_variables:
    # Run Bayesian Optimization
    print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
    result = gp_minimize(
        func=objective,      # Objective function
        dimensions=search_space,  # Search space
        n_calls=50,         # Number of function evaluations
        random_state=42,     # For reproducibility,
        n_initial_points=12
    )

    # Best parameters found
    best_params = dict(zip(['size_gen', 'alpha', 'elite_factor', 'whites'], result.x))
    print("Best Parameters:", best_params)
    print("Best MSE (Min-Max Approach):", result.fun)


    
    ruta = f'EGNABO_params/{v}Osc'
    with open(f'{ruta}.pkl', "wb") as file:
        pickle.dump(best_params, file)


    for keys in best_params.keys():
        print(f'Problema {keys}: {best_params[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 50
##############################################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
IT:  0 	Best cost:  385.77079856789874
	NFEVALS = 375000 F = -0.08274340678672092
	X = [[ 1.25858129e-06 -5.59221961e-04 -9.33710398e-04 -2.19371612e-03
  -3.13276866e-03 -4.85987583e-03 -7.79440948e-03 -1.23897609e-02
  -1.95931749e-02 -2.99373854e-02 -4.51640320e-02 -6.64260761e-02
  -9.37235480e-02 -1.29564434e-01 -1.73898645e-01 -2.26017270e-01
  -2.86806869e-01 -3.54976107e-01 -4.28643729e-01 -5.02492183e-01
  -5.73883395e-01 -6.38610758e-01 -6.91155134e-01 -7.28248929e-01
  -7.48894187e-01 -7.48819249e-01 -7.29047660e-01 -6.90477287e-01
  -6.37340564e-01 -5.73140443e-01 -5.01735977e-01 -4.27101884e-01
  -3.54631171e-01 -2.86627811e-01 -2.25100069e-01 -1.72700006e-01
  -1.28651553e-

	NFEVALS = 375000 F = -0.00028288189368633486
	X = [[-0.54624189  0.08195049 -0.11848384 -0.0618275  -0.03818067 -0.05131097
  -0.05275672 -0.04817176 -0.06000857 -0.13393306 -0.17702437 -0.25721442
  -0.32071037 -0.38090676 -0.44017365 -0.50386642 -0.57194021 -0.62442838
  -0.64704582 -0.64669157 -0.56999206 -0.47952515 -0.3453643  -0.20655715
  -0.05398614  0.11728173  0.27632466  0.38570928  0.48479614  0.57862975
   0.60904445  0.62688683  0.57502811  0.49558642  0.41657694  0.34431107
   0.26905924  0.22221498  0.20326799  0.16595305  0.13440944  0.12071222
   0.09387839  0.07240617  0.08566199  0.05750549  0.03172771  0.01806424
   0.03481944 -0.001725  ]]

  |Psi1> done

F should be approx -4.54e-5
IT:  0 	Best cost:  303.75920922716193
	NFEVALS = 375000 F = -0.00017472330283303355
	X = [[ 0.60906521 -0.04407612  0.20216735  0.13448051  0.12640769  0.18686792
   0.21403931  0.22464536  0.25861311  0.26647692  0.32727289  0.35697094
   0.41414979  0.48137654  0.52299467  0.512144